# Import bibliotek

In [ ]:
import autoroot  # noqa
import joblib
from dotenv import load_dotenv

from grawiki.db.falkordb import FalkorGraphDB
from grawiki.rag.graph_rag import GraphRAG
from pathlib import Path

In [2]:
load_dotenv(override=True)

True

# RAG init

In [3]:
database = FalkorGraphDB(db_path="local_falcor.db", graph_name="grawiki")

In [4]:
rag = GraphRAG(
    db=database,
    model="openrouter:openai/gpt-5-mini",
    embedding_model="openrouter:openai/text-embedding-3-small",
    max_workers=10,
    resolve_entities_on_ingest=True,
    entity_resolution_threshold=0.9,
)

# Data reading / loading

In [5]:
documents = list(Path("experimental_data").glob("*.txt"))
documents

[PosixPath('experimental_data/agent_architectures.txt'),
 PosixPath('experimental_data/react_agents.txt'),
 PosixPath('experimental_data/summary_of_architecutres.txt'),
 PosixPath('experimental_data/reflection_agents.txt')]

In [6]:
first_ingest = [documents[0], documents[2]]
first_ingest

[PosixPath('experimental_data/agent_architectures.txt'),
 PosixPath('experimental_data/summary_of_architecutres.txt')]

# Ingestion

In [7]:
await rag.ingest(first_ingest[0])

In [8]:
await rag.ingest(first_ingest[1])

# Step by step procssing

In [26]:
new_doc_1_pth = documents[1]
new_doc_1 = rag.read_document(new_doc_1_pth)
chunks_1 = rag.chunk_document(new_doc_1)
doc_embed_1 = await rag.embed_document(new_doc_1)
chunk_embed_1 = await rag.embed_chunks(chunks_1)
doc_node_1 = rag.build_document_node(new_doc_1, doc_embed_1)
chunk_node_1 = rag.build_chunk_nodes(chunks_1, chunk_embed_1)
chunk_1_kg = await rag.extract_kg_per_chunk(chunks_1)

/tmp/ipykernel_6307/390348096.py:8: RuntimeWarning: coroutine 'GraphRAG.extract_kg_per_chunk' was never awaited
  chunk_1_kg = await rag.extract_kg_per_chunk(chunks_1)


In [91]:
joblib.dump(chunk_1_kg, "chunk_1_kg.joblib")

['chunk_1_kg.joblib']

# Debug

In [9]:
sim_finder = rag._entity_similarity

In [10]:
duplicates = await sim_finder.find_duplicate_candidates(limit=10, threshold=0.9)

In [11]:
duplicate_candidate_1 = duplicates.similarity_candidates[0]
dupl_source_1 = duplicate_candidate_1.source
dupl_source_1

Node(id='804079f7-6e53-4987-929e-fc68674e9bbe', labels=frozenset({'Class'}), semantic_key='class_messagegraph', name='MessageGraph', properties={}, embedding=[0.0084686279296875, -0.00109100341796875, -0.02911376953125, -0.00689697265625, -0.03216552734375, 0.0244293212890625, -0.0098876953125, 0.0113525390625, -0.01141357421875, 0.0019969940185546875, 0.0158233642578125, -0.018951416015625, -0.006137847900390625, 0.002201080322265625, 0.0347900390625, -0.01885986328125, -0.00841522216796875, 0.01617431640625, 0.0301513671875, 0.03564453125, 0.016082763671875, 0.0660400390625, 0.031036376953125, 0.0382080078125, -0.01535797119140625, 0.01068115234375, -0.03277587890625, -0.0211944580078125, 0.021087646484375, -0.0175628662109375, 0.0213470458984375, -0.0323486328125, 0.000286102294921875, -0.0305633544921875, 0.031646728515625, 0.01178741455078125, -0.009857177734375, 0.0128326416015625, 0.009765625, 0.01061248779296875, 0.003261566162109375, 0.050567626953125, -0.0010919570922851562, 

In [15]:
len(duplicate_candidate_1.hits)

1

In [29]:
report = await rag.dedupe_entities(
    limit=5, threshold=0.95, min_merge_score=0.95, dry_run=True
)

In [30]:
report

[]